# 05 — Assertion Views and Precise Retract

This notebook demonstrates the v0.2 SDK assertion-view path after the namespace migration:

1. Write facts and keep the returned `asrt_id` values.
2. Use `fg.entities.where(...)` for snapshot selection.
3. Use `EntitySnapshot -> AssertionView -> AssertionRecordSet` to select exact assertions.
4. Use `.where(...)`, `.at(...)`, `.by_id(...)`, and `.one()` before destructive actions; use `_meta={...}` for metadata filters such as version.
5. Create a frozen assertion view with `fg.assertion_views.create(..., asrt_ids=...)` / `asrts=...`.
6. Read the frozen view back through `fg.assertions.by_ids(view.asrt_ids)`.
7. Keep frozen assertion views separate from entity-query call-site options.

Every code cell asserts on the behavior it demonstrates.

## 0. Imports

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_src = cwd / "src"
if not repo_src.exists() and cwd.name == "examples":
    repo_src = cwd.parent / "src"
if repo_src.exists() and str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

In [ ]:
from factgraph.sdk import Entity, Field, Identity, SDKStore, SDKSchemaError, SDKStoreError


## 1. Define a small schema

`User.user_id` is the primary identity. `User.locale` is a secondary coordinate dimension with a default. `name` is single-valued and `tag` is multi-valued.

In [ ]:
class User(Entity):
    user_id: str = Identity()
    locale: str = Identity()
    name: str = Field()
    tag: list[str] = Field()

## 2. Write facts and keep assertion ids

`set(...)` and `add(...)` return persisted assertion ids. Those ids are the precise handles used later for frozen views and `retract(...)`.

In [ ]:
fg = SDKStore([User])
ref = fg.entities.create(User, user_id="u-1", locale="en")

ids = {
    "name_v1": fg.fields.set(
        User.name,
        ref,
        "Alice",
        meta={
            "source": "seed",
            "trace_id": "import-001",
            "note": "seed import",
            "version": "name-v1",
            "valid_from": "2026-01-01T00:00:00Z",
            "valid_to": "2026-02-01T00:00:00Z",
        },
    ),
    "name_v2": fg.fields.set(
        User.name,
        ref,
        "Alicia",
        meta={
            "source": "correction",
            "trace_id": "manual-001",
            "note": "manual correction",
            "version": "name-v2",
            "valid_from": "2026-02-01T00:00:00Z",
        },
    ),
    "tag_vip": fg.fields.add(
        User.tag,
        ref,
        "vip",
        meta={
            "source": "seed",
            "trace_id": "import-001",
            "note": "review label",
            "version": "tag-v1",
            "valid_from": "2026-01-10T00:00:00Z",
            "valid_to": "2026-03-01T00:00:00Z",
        },
    ),
    "tag_legacy": fg.fields.add(
        User.tag,
        ref,
        "legacy-no-valid-from",
        meta={"source": "legacy", "note": "legacy import", "version": "tag-v0"},
    ),
}

assert set(ids) == {"name_v1", "name_v2", "tag_vip", "tag_legacy"}
assert all(isinstance(value, str) and value for value in ids.values())
ids

## 3. Use `fg.entities.where(...)` for snapshot selection

Entity queries return snapshots. Assertion metadata stays on assertion views, so version/source/note filters use `fg.assertions.where(...)` or `snapshot.assertions.where(...)` with `_meta={...}`.

In [ ]:
matches = fg.entities.where(User, name="Alicia")
assert len(matches) == 1
assert matches[0].name == "Alicia"
assert not hasattr(matches[0], "version")

versioned_record = fg.assertions.where(
    field=User.name,
    e_ref=ref,
    _meta={"version": "name-v2", "note": "manual correction"},
).one()
assert versioned_record.value == "Alicia"
assert versioned_record.meta.raw["version"] == "name-v2"


## 4. Read a snapshot and inspect assertion records

Scalar fields are good for display; assertion collections are the path for audit, selection, view creation, and retract.

In [ ]:
snap = fg.entities.get(User, user_id="u-1", locale="en")
assert snap is not None

assert snap.identity == {"user_id": "u-1", "locale": "en"}
assert snap.identity_available is True
assert isinstance(snap.name, str)

name_history = snap.field("name").all
same_path = snap.assertions.name.all

assert tuple(record.asrt_id for record in name_history) == tuple(
    record.asrt_id for record in same_path
)
assert hasattr(name_history, "where")
assert hasattr(name_history, "at")
assert not hasattr(name_history, "version")
assert hasattr(name_history, "by_id")

## 5. Use `where`, `at`, `_meta`, and `by_id` as assertion filters

`.at(t)` is a business-time point filter over `valid_from` / `valid_to`, not an ingest-time filter. `valid_to` is exclusive.

In [ ]:
january_name = (
    snap.field("name")
    .all
    .where(_meta={"source": "seed", "trace_id": "import-001", "version": "name-v1"})
    .at("2026-01-15T00:00:00Z")
    .by_id(ids["name_v1"])
    .one()
)

assert january_name.value == "Alice"
assert january_name.meta.raw["valid_from"] == "2026-01-01T00:00:00Z"
assert january_name.meta.raw["valid_to"] == "2026-02-01T00:00:00Z"

february_name = snap.field("name").all.at("2026-02-01T00:00:00Z").where(_meta={"version": "name-v2"}).one()
assert february_name.value == "Alicia"

march_boundary = snap.field("tag").all.at("2026-03-01T00:00:00Z")
assert march_boundary.where(value="vip").all() == ()

missing_valid_from = snap.field("tag").all.by_id(ids["tag_legacy"])
assert missing_valid_from.at("2026-01-15T00:00:00Z").all() == ()


## 6. Create a frozen assertion view

A frozen assertion view names an immutable set of assertion ids. It can be created from ids or from objects exposing `.asrt_id`; only the ids become membership. There is no built-in `default` view, and the name `"default"` is not reserved.

In [ ]:
review_target = snap.field("name").all.where(value="Alice", _meta={"source": "seed"}).one()
review_tag = snap.field("tag").all.where(value="vip", _meta={"source": "seed"}).one()

view = fg.assertion_views.create("review_set", asrts=[review_target, review_tag])
default_named_view = fg.assertion_views.create("default", asrt_ids=[review_tag.asrt_id])

assert type(view).__name__ == "FrozenAssertionSet"
assert view.name == "review_set"
assert isinstance(view.asrt_ids, frozenset)
assert view.asrt_ids == frozenset({ids["name_v1"], ids["tag_vip"]})
assert fg.assertion_views.get("review_set") is view
assert fg.assertion_views.get("default") is default_named_view
assert default_named_view.asrt_ids == frozenset({ids["tag_vip"]})


## 7. Read a frozen view back through `fg.assertions`

Frozen views are membership sets. To inspect the actual assertion records, use `fg.assertions.by_ids(view.asrt_ids)`.

In [ ]:
records = fg.assertions.by_ids(view.asrt_ids)

assert {record.asrt_id for record in records} == {ids["name_v1"], ids["tag_vip"]}
assert records.where(value="Alice").one().asrt_id == ids["name_v1"]
assert records.where(value="vip").one().asrt_id == ids["tag_vip"]
assert fg.assertions.by_id(ids["name_v1"]).value == "Alice"
assert fg.assertions.by_id("missing-asrt") is None

## 8. Keep frozen views separate from entity queries

The old keyword named `view` is not an entity-query option, and `policy=` is removed from the current public read surface. Use `fg.assertions.by_ids(...)` for record-level view readback; use assertion-level `_meta={...}` filters for assertion metadata.

In [ ]:
try:
    fg.entities.where(User, **{"view": "review_set"})
except (SDKSchemaError, SDKStoreError) as exc:
    view_message = str(exc)
else:
    raise AssertionError("the old view keyword must be rejected on entities.where")

assert "view" in view_message

try:
    fg.entities.where(User, policy=view)
except SDKStoreError as exc:
    policy_message = str(exc)
else:
    raise AssertionError("policy= must be rejected on the current read surface")

assert "removed" in policy_message


## 9. Precise retract uses `target.asrt_id`

Do not retract by value or by `records[0]`. Use the read-side selectors to prove there is exactly one target, then pass its `asrt_id` to `fg.assertions.retract(...)`.

In [ ]:
target = (
    fg.assertions.by_ids(view.asrt_ids)
    .where(value="Alice", _meta={"source": "seed", "trace_id": "import-001"})
    .one()
)

revoker_asrt_id = fg.assertions.retract(
    target.asrt_id,
    meta={"source": "manual-fix", "trace_id": "fix-001"},
)
assert isinstance(revoker_asrt_id, str) and revoker_asrt_id

post_retract = fg.assertions.by_id(target.asrt_id)
assert post_retract is not None
assert post_retract.is_active is False

snap_after = fg.entities.get(User, user_id="u-1", locale="en")
assert snap_after is not None
assert snap_after.field("name").all.by_id(target.asrt_id).one().is_active is False


## 10. Summary

The professional workflow is:

```text
write returns asrt_id
  -> entity queries choose snapshots
  -> snapshot exposes AssertionView
    -> where/at/_meta/by_id narrow the assertion set
      -> one() proves exactly one target
        -> views can freeze target ids
          -> fg.assertions.by_ids(...) reads them back
            -> fg.assertions.retract(target.asrt_id) performs the mutation
```

This keeps assertion selection on the read side and assertion-id mutation on the Layer 3 namespace.